In [13]:
# Import libraries for data preparation
import pandas as pd
import json
from pathlib import Path

# Define input and output paths
DATA_DIR = Path("processed")

INPUT_FILE = DATA_DIR / "bls_occupational_projections_clean.csv"
OUTPUT_FILE = DATA_DIR / "bls_employment.json"

# Load the cleaned BLS dataset
df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "occupation_code": "string",
        "parent_code": "string"
    }
)

# Check the dataset
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Rows: 1117
Columns: 26


,2025_national_employment_matrix_title,2025_national_employment_matrix_code,occupation_type,employment_2025,employment_2035,employment_distribution_percent_2025,employment_distribution_percent_2035,employment_change_numeric_2025_35,employment_change_percent_2025_35,percent_self_employed_2025,...,related_occupational_outlook_handbook_ooh_content_1,occupation_code,occupation_title,soc_level,expected_parent_code,parent_code,missing_parent,soc_official_title,soc_match,hierarchy_source
0,"Total, all occupations",00-0000,Summary,"170,280.8","176,198.5",100.0,100.0,"5,917.6",3.5,5.8,...,—,00-0000,"Total, all occupations",major,NaN,<NA>,False,NaN,False,unresolved
1,Management occupations,11-0000,Summary,"13,683.4","14,527.6",8.0,8.2,844.2,6.2,14.4,...,—,11-0000,Management occupations,major,NaN,<NA>,False,Management Occupations,True,soc_2018
2,Top executives,11-1000,Summary,"3,904.1","4,095.2",2.3,2.3,191.1,4.9,2.7,...,—,11-1000,Top executives,minor,11-0000,11-0000,False,Top Executives,True,soc_2018
3,Chief executives,11-1011,Line item,291.6,300.9,0.2,0.2,9.3,3.2,26.6,...,"<a href=""https://www.bls.gov/ooh/management/to...",11-1011,Chief executives,detailed,11-1010,11-1000,False,Chief Executives,True,soc_2018
4,General and operations managers,11-1021,Line item,"3,599.0","3,780.3",2.1,2.1,181.3,5.0,0.8,...,"<a href=""https://www.bls.gov/ooh/management/to...",11-1021,General and operations managers,detailed,11-1020,11-1000,False,General and Operations Managers,True,soc_2018


In [15]:
# Find employment columns for 2025 and 2035
employment_columns = [
    col for col in df.columns
    if "employment" in col.lower()
]

print("Employment columns:")
for col in employment_columns:
    print(col)

Employment columns:
2025_national_employment_matrix_title
2025_national_employment_matrix_code
employment_2025
employment_2035
employment_distribution_percent_2025
employment_distribution_percent_2035
employment_change_numeric_2025_35
employment_change_percent_2025_35


In [17]:
# Define the columns needed for the chart
CODE_COL = "occupation_code"
NAME_COL = "occupation_title"
EMP_2025_COL = "employment_2025"
EMP_2035_COL = "employment_2035"
EMP_NUM_CHANGE_2025_35_COL = "employment_change_numeric_2025_35"
EMP_PERC_CHANGE_2025_35_COL = "employment_change_percent_2025_35"

# Select only the required columns
chart_df = df[
    [CODE_COL, NAME_COL, EMP_2025_COL, EMP_2035_COL, EMP_NUM_CHANGE_2025_35_COL,EMP_PERC_CHANGE_2025_35_COL]
].copy()

# Rename columns for consistent use in JavaScript
chart_df.columns = [
    "code",
    "name",
    "employment_2025",
    "employment_2035",
    "employment_change_numeric_2025_35",
    "employment_change_percent_2025_35"
]

# Convert employment and change columns to numeric values
numeric_cols = [
    "employment_2025",
    "employment_2035",
    "employment_change_numeric_2025_35",
    "employment_change_percent_2025_35"
]

# Convert employment columns to numeric values
for col in numeric_cols:
    chart_df[col] = pd.to_numeric(
        chart_df[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce"
    )

# Check for missing values
print(chart_df.isna().sum())

display(chart_df.head())

code                                 5
name                                 0
employment_2025                      5
employment_2035                      5
employment_change_numeric_2025_35    5
employment_change_percent_2025_35    5
dtype: int64


,code,name,employment_2025,employment_2035,employment_change_numeric_2025_35,employment_change_percent_2025_35
0,00-0000,"Total, all occupations",170280.8,176198.5,5917.6,3.5
1,11-0000,Management occupations,13683.4,14527.6,844.2,6.2
2,11-1000,Top executives,3904.1,4095.2,191.1,4.9
3,11-1011,Chief executives,291.6,300.9,9.3,3.2
4,11-1021,General and operations managers,3599.0,3780.3,181.3,5.0


In [19]:
# Keep only records with a valid occupation code
chart_df = chart_df[
    chart_df["code"].astype("string").str.fullmatch(
        r"\d{2}-\d{4}", na=False
    )
].copy()

# Remove records with missing employment values
chart_df = chart_df.dropna(
    subset=["employment_2025", "employment_2035"]
)

# Ensure every occupation appears only once
assert not chart_df["code"].duplicated().any(), \
    "Duplicate occupation codes found"

# Sort by 2025 employment, then 2035 employment
chart_df = chart_df.sort_values(
    by=["employment_2025", "employment_2035"],
    ascending=[False, False]
)

print("Records ready for chart:", len(chart_df))
display(chart_df.head(10))

Records ready for chart: 1112


,code,name,employment_2025,employment_2035,employment_change_numeric_2025_35,employment_change_percent_2025_35
0,00-0000,"Total, all occupations",170280.8,176198.5,5917.6,3.5
675,43-0000,Office and administrative support occupations,18792.7,18040.5,-752.1,-4.0
642,41-0000,Sales and related occupations,14533.2,14326.4,-206.8,-1.4
1045,53-0000,Transportation and material moving occupations,14139.0,14740.4,601.4,4.3
556,35-0000,Food preparation and serving related occupations,13826.5,14429.1,602.5,4.4
1,11-0000,Management occupations,13683.4,14527.6,844.2,6.2
49,13-0000,Business and financial operations occupations,11368.5,11997.1,628.6,5.5
410,29-0000,Healthcare practitioners and technical occupat...,10232.8,11052.4,819.5,8.0
267,25-0000,Educational instruction and library occupations,9907.4,10049.4,142.0,1.4
905,51-0000,Production occupations,8780.5,8747.1,-33.5,-0.4


In [21]:
# Convert the prepared DataFrame to JSON records
chart_data = chart_df.to_dict(orient="records")

# Save the chart data as JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(chart_data, f, ensure_ascii=False, indent=2)

# Verify the output
print("JSON saved:", OUTPUT_FILE)
print("Number of occupations:", len(chart_data))

JSON saved: processed\bls_employment.json
Number of occupations: 1112
